# Generation - RAG-Based Mental Health Support Chatbot

# 1. Environment Setup

In [2]:
!pip install -q langchain-groq langchain-core

In [31]:
import os
from getpass import getpass
from google.colab import userdata

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

from IPython.display import display, Markdown


In [5]:
# GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [11]:
if "GROQ_API_KEY" not in os.environ:
    print("Enter your Groq API Key:")
    os.environ["GROQ_API_KEY"] = getpass()

Enter your Groq API Key:
··········


# 2. Groq Initialization

In [23]:
def initialize_llm(model_name="llama-3.1-8b-instant", temperature=0.15, max_tokens=500, top_p=0.90):
    llm = ChatGroq(
        model_name=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        model_kwargs={
            "top_p": top_p
        }
    )

    print(f"Successfully initialized {model_name} on Groq.")
    return llm

In [24]:
llm_engine = initialize_llm()

Successfully initialized llama-3.1-8b-instant on Groq.


# 3. Prompt Template

In [27]:
def create_prompt_template():
    system_template = """You are an empathetic, professional, and highly supportive mental health assistant.
    Your goal is to provide grounded, safe, and validating responses to users seeking support.

    CRITICAL DIRECTIVES - YOU MUST OBEY THESE RULES:
    1. NO DIAGNOSIS: Never tell a user they have a specific psychological or medical condition.
    2. NO PRESCRIPTION: Never suggest, recommend, or discuss medications or physical treatments.
    3. GROUNDING: You must formulate your advice STRICTLY based on the provided [Context] block. Do not hallucinate external facts.
    4. CONTEXTUAL SILENCE: If the provided [Context] does not contain relevant information to answer the query, politely state that you do not have specific resources on that topic, but offer general empathetic listening.
    5. CRISIS PROTOCOL: If the user mentions suicide, self-harm, or severe abuse, immediately stop providing standard advice. Respond ONLY with high empathy and instruct them to contact emergency services or a crisis hotline immediately.
    6. EMOTIONAL INTELLIGENCE: The user's detected emotion is provided below. Adapt your tone to be highly sensitive to this specific emotion.
    7. LANGUAGE: You MUST respond in the language specified in the [Target Language] field.
    8. FORMATTING: Use Markdown formatting. Use bold text to emphasize key calming techniques, and use bullet points when listing steps.

    [Target Language]: {language}
    [Detected Emotion]: {emotion}

    [Context]:
    {context}
    """

    user_template = """User Query: {user_query}"""

    prompt = ChatPromptTemplate.from_messages([
        SystemMessagePromptTemplate.from_template(system_template),
        HumanMessagePromptTemplate.from_template(user_template)
    ])

    print("Prompt Template successfully created")
    return prompt

In [28]:
prompt_template = create_prompt_template()

Prompt Template successfully created


# 4. LCEL Chain

In [18]:
mock_inputs = {
    "language": "English",
    "emotion": "fear",
    "context": "Context Chunk 1: Breathing exercises, such as the 4-7-8 method, have been shown to reduce acute anxiety. \nContext Chunk 2: Grounding techniques like naming 5 things you can see help distract the mind from panic.",
    "user_query": "I am having a panic attack about my exams tomorrow, I don't know what to do."
}


In [29]:
output_parser = StrOutputParser()

generation_chain = prompt_template | llm_engine | output_parser

response = generation_chain.invoke(mock_inputs)

print("=== Chatbot Response ===")
print(response)

=== Chatbot Response ===
**You're not alone in this feeling.** It's completely normal to feel overwhelmed before a big exam. Let's take a step back and focus on calming your body and mind.

**Grounding techniques can be really helpful in this situation.** Try to focus on your surroundings and name 5 things you can see right now. This can help distract you from your anxiety and bring you back to the present moment.

* Take a deep breath in through your nose for a count of 4
* Hold your breath for a count of 7
* Slowly exhale through your mouth for a count of 8
* Repeat this process a few times to help calm your body

**Remember, you've prepared well for your exams, and you have the skills and knowledge to succeed.** Try to focus on positive thoughts and remind yourself of your strengths.

If you feel like you're getting overwhelmed, take a break and do something that helps you relax, like going for a short walk or listening to calming music.

Keep in mind that it's okay to feel scared o

In [32]:
display(Markdown(response))

**You're not alone in this feeling.** It's completely normal to feel overwhelmed before a big exam. Let's take a step back and focus on calming your body and mind.

**Grounding techniques can be really helpful in this situation.** Try to focus on your surroundings and name 5 things you can see right now. This can help distract you from your anxiety and bring you back to the present moment.

* Take a deep breath in through your nose for a count of 4
* Hold your breath for a count of 7
* Slowly exhale through your mouth for a count of 8
* Repeat this process a few times to help calm your body

**Remember, you've prepared well for your exams, and you have the skills and knowledge to succeed.** Try to focus on positive thoughts and remind yourself of your strengths.

If you feel like you're getting overwhelmed, take a break and do something that helps you relax, like going for a short walk or listening to calming music.

Keep in mind that it's okay to feel scared or anxious, but try not to let it consume you. You got this!

# 5. Routing Wrapper

In [33]:
def generate_safe_response(input_data, chain):
    intent = input_data.get("intent", "asking_mental_health_question")
    context = input_data.get("context", "")

    # === Crisis Routing ===
    if intent == "crisis" or "suicide" in input_data.get("user_query", "").lower():
        return (
            "🚨 **CRISIS ALERT:** Your safety is the absolute priority. "
            "I am an AI and cannot provide the help you need right now. "
            "Please immediately contact emergency services or a crisis hotline in your area. "
            "You are not alone, and help is available."
        )

    # === Out of Scope ===
    if intent == "out_of_scope":
        return (
            "I am a mental health support assistant. "
            "I'm afraid I cannot help with that specific topic, but I am here if you need to talk about your well-being."
        )

    # === Missing Context ===
    if not context or context.strip() == "":
        return (
            "I'm here to listen and support you. However, I don't have specific resources "
            "or techniques on that exact topic in my knowledge base right now. "
            "Could you tell me a bit more about what you're experiencing?"
        )

    # === Main Execution ===
    try:
        # If it passes all safety gates, send it to the LLM
        response = chain.invoke(input_data)
        return response

    except Exception as e:
        # If Groq times out or the internet disconnects
        print(f"[API ERROR]: {e}")
        return (
            "I'm currently experiencing a technical connection issue and cannot process your request right now. "
            "Please try again in a moment. If you are in distress, please reach out to a human support line."
        )


In [35]:
print("=== TEST: The Crisis Intent ===")

crisis_mock = {
    "intent": "crisis",
    "user_query": "I can't take this anymore, I want to end it."
}

crisis_response = generate_safe_response(crisis_mock, generation_chain)
display(Markdown(crisis_response))

=== TEST: The Crisis Intent ===


🚨 **CRISIS ALERT:** Your safety is the absolute priority. I am an AI and cannot provide the help you need right now. Please immediately contact emergency services or a crisis hotline in your area. You are not alone, and help is available.

In [36]:
print("=== TEST: The Suicide Intent ===")

suicide_mock = {
    "intent": "asking_mental_health_question",
    "user_query": "I can't take this anymore, I want to commit suicide."
}

suicide_response = generate_safe_response(suicide_mock, generation_chain)
display(Markdown(suicide_response))

=== TEST: The Suicide Intent ===


🚨 **CRISIS ALERT:** Your safety is the absolute priority. I am an AI and cannot provide the help you need right now. Please immediately contact emergency services or a crisis hotline in your area. You are not alone, and help is available.

In [37]:
print("\n=== TEST: Missing Context ===")

empty_context_mock = {
    "intent": "asking_mental_health_question",
    "user_query": "What is the best type of mattress for anxiety?",
    "context": "" # Empty
}

empty_context_response = generate_safe_response(empty_context_mock, generation_chain)
display(Markdown(empty_context_response))


=== TEST: Missing Context ===


I'm here to listen and support you. However, I don't have specific resources or techniques on that exact topic in my knowledge base right now. Could you tell me a bit more about what you're experiencing?

In [38]:
print("\n=== TEST: The Normal Execution ===")

normal_response = generate_safe_response(mock_inputs, generation_chain)
display(Markdown(normal_response))


=== TEST: The Normal Execution ===


**You're not alone in this feeling.** It's completely normal to feel overwhelmed before a big exam. Let's take a few deep breaths together and try to calm down.

**The 4-7-8 method can help**. Have you tried it before? It's a simple breathing exercise that can reduce anxiety. Here's how it works:

* Inhale through your nose for a count of 4
* Hold your breath for a count of 7
* Exhale through your mouth for a count of 8

Try to focus on your breath and let go of any thoughts about the exam. Remember, you've prepared well, and you're ready for this.

**Grounding techniques can also help**. Let's try to distract your mind from the panic. Can you tell me:

* What's the first thing you see right in front of you?
* What's the next thing you notice?
* Keep going until you've named 5 things you can see.

This can help you focus on the present moment and calm down. Take your time, and remember that you're safe.

If you need to talk more or want to try something else, I'm here to listen and support you.